In [ ]:
ReportFolderName = 'BERT-Based_Proposed_Version_without_normalization'

In [ ]:
import shutil
import os

# Keywords for folders to delete
folders_to_delete = ["logs", ReportFolderName, "results", "sample_data"]

# Delete matching folders
for item in os.listdir("."):
    if os.path.isdir(item) and any(keyword in item for keyword in folders_to_delete):
        shutil.rmtree(item)
        print(f"✅ Deleted folder: {item}")

# # Delete all files in the current directory
# for item in os.listdir("."):
#     if os.path.isfile(item):
#         os.remove(item)
#         print(f"🗑️ Deleted file: {item}")

print("\n🎯 Full cleanup completed. All matching folders and all files removed.")

✅ Deleted folder: sample_data

🎯 Full cleanup completed. All matching folders and all files removed.


In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

!pip install -q --upgrade transformers datasets peft accelerate scikit-learn tqdm "torchao>=0.16.0"

import torch, string, numpy as np, pandas as pd
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments
)
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.5 MB/s eta 0:00:00


In [ ]:
random_state = 44

# Seed everything for reproducibility (fix: random_state was unused before)
import os, random
import numpy as np
import torch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(random_state)


In [ ]:
import re

def pre_process_sms(text):
    # text = re.sub(r"http\\S+", "URL", text)
    # Bangladeshi mobile numbers (optional +88 country code, optional separators)
    # text = re.sub(r'(\\+?88)?[\\s-]?01[3-9][\\s-]?\\d{4}[\\s-]?\\d{4}', 'PHONE', text)
    # Generic phone-like number sequences in the English variant (7+ digits, optional separators)
    # text = re.sub(r'(?<!\\d)(\\+?\\d[\\d\\s-]{6,}\\d)(?!\\d)', 'PHONE', text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()

def preprocess_text(batch):
    batch["text"] = pre_process_sms(batch["text"])
    return batch


In [ ]:
label2id = {"normal": 0, "promo": 1, "smish": 2}
id2label = {v: k for k, v in label2id.items()}

def encode_labels(batch):
    batch["label"] = label2id[batch["label"]]
    return batch

In [ ]:
dataset = load_dataset("shariul-islam/bengali-sms-smishing-dataset")

README.md:   0%|          | 0.00/3.03k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/404k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/60.6k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/118k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4903 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/701 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1401 [00:00<?, ? examples/s]

In [ ]:
# TEST SAMPLE DATA SET (DISABLED by default)
# Set USE_SAMPLE_DATA = True ONLY when you want to smoke-test on the small local CSVs.
# When False, the real HuggingFace dataset loaded in the previous cell is used.
USE_SAMPLE_DATA = False

if USE_SAMPLE_DATA:
    from datasets import Dataset, DatasetDict
    import pandas as pd

    # Load the three sample CSVs (upload them to Colab first)
    dataset = DatasetDict({
        "train":      Dataset.from_pandas(pd.read_csv("sample_train.csv")),
        "validation": Dataset.from_pandas(pd.read_csv("sample_val.csv")),
        "test":       Dataset.from_pandas(pd.read_csv("sample_test.csv")),
    })
    print("⚠️ Using SAMPLE data — not the full HF dataset.")
else:
    print("✅ Using the full HuggingFace dataset.")


✅ Using the full HuggingFace dataset.


In [ ]:
dataset = dataset.map(encode_labels)
dataset = dataset.map(preprocess_text)

Map:   0%|          | 0/4903 [00:00<?, ? examples/s]

Map:   0%|          | 0/701 [00:00<?, ? examples/s]

Map:   0%|          | 0/1401 [00:00<?, ? examples/s]

Map:   0%|          | 0/4903 [00:00<?, ? examples/s]

Map:   0%|          | 0/701 [00:00<?, ? examples/s]

Map:   0%|          | 0/1401 [00:00<?, ? examples/s]

In [ ]:
dataset['train'][2]

{'label': 1,
 'text': 'bonus soho 2gb45tk3din dial 1215534 or mygplimy',
 'source': 'Banglish'}

In [ ]:
train_dataset = dataset['train']
val_dataset = dataset['validation']
test_dataset = dataset['test']

In [ ]:
train_df = pd.DataFrame(train_dataset)
print(train_df["source"].value_counts())

source
Banglish    1261
Bengali     1253
English     1214
CodeMix     1175
Name: count, dtype: int64


In [ ]:
test_df = pd.DataFrame(test_dataset)
print(test_df["source"].value_counts())

source
Banglish    360
Bengali     358
English     347
CodeMix     336
Name: count, dtype: int64


In [ ]:
def save_model_into_huggingface(model, tokenizer, model_alias):
  # ── Save LoRA adapter to Hugging Face Hub ───────────────────────
  from huggingface_hub import HfApi

  HF_USERNAME = "shariul-islam"   # your HF username
  repo_id = f"{HF_USERNAME}/proposed-nourl-{model_alias.lower().replace('/', '-')}"

  # Create the repo if it doesn't exist
  from huggingface_hub import create_repo
  try:
      create_repo(repo_id, repo_type="model", private=False, exist_ok=True)
      print(f"✅ Repo ready: {repo_id}")
  except Exception as e:
      print(f"Repo note: {e}")

  # Save adapter locally first, then push
  adapter_local_path = f"./adapters/{model_alias}"
  model.save_pretrained(adapter_local_path)
  tokenizer.save_pretrained(adapter_local_path)

  # Push to Hub
  model.push_to_hub(repo_id, commit_message=f"Add LoRA adapter: {model_alias}")
  tokenizer.push_to_hub(repo_id, commit_message=f"Add tokenizer: {model_alias}")
  print(f"✅ Pushed to HF: https://huggingface.co/{repo_id}")

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize


def evaluate_and_report(trainer, test_dataset, label2id, model_alias, include_source=False):
    """
    Generates and saves evaluation reports for a trained model:
      - Classification report (text, CSV, LaTeX)
      - Confusion matrix (overall + per-source)
      - ROC curve (multi-class one-vs-rest)
      - Appends model summary to ./reports/summary.csv

    All outputs saved directly in ./reports/ (no per-model subfolders)
    """

    print(f"\n📊 Generating Evaluation Report for {model_alias}")

    # -----------------------------
    # 1️⃣ Prepare directory
    # -----------------------------
    report_dir = f"./{ReportFolderName}"
    os.makedirs(report_dir, exist_ok=True)

    # -----------------------------
    # 2️⃣ Predictions
    # -----------------------------
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = test_dataset["label"]

    np.save(f"{report_dir}/{model_alias}_y_true.npy", y_true)
    np.save(f"{report_dir}/{model_alias}_y_pred.npy", y_pred)

    pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).to_csv(
    f"{report_dir}/{model_alias}_predictions.csv", index=False)

    class_names = list(label2id.keys())

    # -----------------------------
    # 3️⃣ Classification Report (OVERALL)
    # -----------------------------
    report_text = classification_report(y_true, y_pred, target_names=class_names)
    # NOTE: this overall dict is kept intact and used for summary.csv at step 7 (fix #9)
    overall_report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

    # Text file
    with open(f"{report_dir}/{model_alias}_classification_report.txt", "w") as f:
        f.write(report_text)

    # CSV file
    df_report = pd.DataFrame(overall_report_dict).transpose().round(4)
    df_report.to_csv(f"{report_dir}/{model_alias}_classification_report.csv")

    print(report_text)

    # -----------------------------
    # 4️⃣ Confusion Matrix
    # -----------------------------
    cm = confusion_matrix(y_true, y_pred, labels=list(label2id.values()))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - {model_alias}")
    plt.tight_layout()
    plt.savefig(f"{report_dir}/{model_alias}_confusion_matrix.png")
    plt.close()

    # -----------------------------
    # 5️⃣ ROC Curve (One-vs-Rest)
    # -----------------------------
    try:
        y_true_bin = label_binarize(y_true, classes=list(label2id.values()))
        y_score = torch.softmax(torch.tensor(predictions.predictions), dim=1).numpy()

        plt.figure(figsize=(6, 5))
        for i, class_name in enumerate(class_names):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=2, label=f"{class_name} (AUC = {roc_auc:.2f})")

        plt.plot([0, 1], [0, 1], "k--", label="Random")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve - {model_alias}")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(f"{report_dir}/{model_alias}_roc_curve.png")
        plt.close()
    except Exception as e:
        print(f"⚠️ Skipping ROC curve for {model_alias}: {e}")

    # -----------------------------
    # 6️⃣ Per-Source Evaluation (Confusion Matrix + Classification Report)
    # NOTE: uses report_dict_src (NOT report_dict) so the overall report is preserved (fix #9)
    # -----------------------------
    if include_source and "source" in test_dataset.column_names:
        sources = test_dataset["source"]
        all_source_reports = []  # store metrics for summary

        for src in set(sources):
            mask = [s == src for s in sources]
            y_true_src = np.array(y_true)[mask]
            y_pred_src = np.array(y_pred)[mask]

            cm_src = confusion_matrix(y_true_src, y_pred_src, labels=list(label2id.values()))
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm_src, annot=True, fmt="d", cmap="Blues",
                        xticklabels=class_names, yticklabels=class_names)
            plt.xlabel("Predicted")
            plt.ylabel("True")
            plt.title(f"Confusion Matrix - {model_alias} ({src})")
            plt.tight_layout()
            plt.savefig(f"{report_dir}/{model_alias}_confusion_matrix_{src}.png")
            plt.close()

            # --- Classification Report (per source) ---
            report_dict_src = classification_report(
                y_true_src, y_pred_src,
                labels=list(label2id.values()),
                target_names=class_names,
                output_dict=True,
                zero_division=0
            )
            report_df = pd.DataFrame(report_dict_src).transpose().round(4)
            report_df.to_csv(f"{report_dir}/{model_alias}_classification_report_{src}.csv", index=True)

            # Add macro averages for summary
            all_source_reports.append({
                "source": src,
                "precision": round(report_dict_src["macro avg"]["precision"], 4),
                "recall": round(report_dict_src["macro avg"]["recall"], 4),
                "f1_score": round(report_dict_src["macro avg"]["f1-score"], 4)
            })

        # --- Summary Report Across Sources ---
        summary_df = pd.DataFrame(all_source_reports)
        summary_df.loc["Average"] = summary_df.mean(numeric_only=True)
        summary_df.to_csv(f"{report_dir}/{model_alias}_source_summary_report.csv", index=False)

        print("\n✅ Per-source classification reports saved.")
        print(f"✅ Summary report saved to: {report_dir}/{model_alias}_source_summary_report.csv")

    # -----------------------------
    # 7️⃣ Summary CSV (append) — reads from the OVERALL report (fix #9)
    # -----------------------------
    acc = round(overall_report_dict["accuracy"], 4)
    precision = round(overall_report_dict["weighted avg"]["precision"], 4)
    recall = round(overall_report_dict["weighted avg"]["recall"], 4)
    f1 = round(overall_report_dict["weighted avg"]["f1-score"], 4)

    summary_dict = {
        "model": model_alias,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

    summary_path = os.path.join(report_dir, "summary.csv")
    if os.path.exists(summary_path):
        existing = pd.read_csv(summary_path)
        existing = pd.concat([existing, pd.DataFrame([summary_dict])], ignore_index=True)
        existing.to_csv(summary_path, index=False)
    else:
        pd.DataFrame([summary_dict]).to_csv(summary_path, index=False)

    print(f"\n✅ {model_alias} → Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print(f"✅ All reports saved in {report_dir}")

    return summary_dict


In [ ]:
def tokenize(batch):
    # Dynamic padding handled by DataCollatorWithPadding -> no padding here (faster, fix #5)
    tokenized = tokenizer(batch["text"], truncation=True, max_length=128)
    tokenized["label"] = batch["label"]
    return tokenized


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [ ]:
reports_dir = f"./{ReportFolderName}"
os.makedirs(reports_dir, exist_ok=True)


In [ ]:
# ============================================
# 2️⃣ DEFINE BASE MODELS
# ============================================

base_models = {
    "mBERT": "bert-base-multilingual-cased",
    "XLM-RoBERTa": "xlm-roberta-base",
    'Muril': 'google/muril-large-cased',
    'Distil-mBERT': 'distilbert-base-multilingual-cased',
}

meta_train_features = []
meta_test_features = []
all_model_results = []

# ============================================
# 3️⃣ LOOP THROUGH EACH BASE MODEL
# ============================================

for model_alias, model_name in base_models.items():
    print(f"\n🔥 Fine-tuning Base Model: {model_alias}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Tokenize into LOCAL variables so re-running with a different model does NOT
    # reuse a previous tokenizer's columns (fix #4). The global *_dataset stays raw.
    train_tok = train_dataset.map(tokenize, batched=True)
    val_tok   = val_dataset.map(tokenize, batched=True)
    test_tok  = test_dataset.map(tokenize, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id
    )

    target_modules = ["query", "key", "value", "dense"]
    if model_name == 'distilbert-base-multilingual-cased':
        target_modules = ["attention.q_lin", "attention.k_lin", "attention.v_lin", "attention.out_lin"]  # DistilBERT names

    # LoRA configuration
    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS"
    )
    print(target_modules)
    model = get_peft_model(base_model, lora_config)

    training_args = TrainingArguments(
        output_dir=f"./results_{model_alias}",
        learning_rate=3e-5,  # 2e-5
        per_device_train_batch_size=32,  # 16, 32, 64
        per_device_eval_batch_size=32,   # 16, 32, 64
        num_train_epochs=10,
        weight_decay=0.01,
        # max_steps=20,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir="./logs",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1",
        greater_is_better=True,
        fp16=True,
        seed=random_state,   # seed belongs here, NOT in Trainer (fix #1)
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,  # removed invalid seed= kwarg (fix #1)
    )

    trainer.train()

    save_model_into_huggingface(model, tokenizer, model_alias)

    # Collect softmax probabilities for stacking
    preds_train = trainer.predict(train_tok)
    preds_test  = trainer.predict(test_tok)

    meta_train_features.append(torch.softmax(torch.tensor(preds_train.predictions), dim=1).numpy())
    meta_test_features.append(torch.softmax(torch.tensor(preds_test.predictions), dim=1).numpy())

    # Evaluate model and save reports
    metrics = evaluate_and_report(trainer, test_tok, label2id, model_alias, include_source=True)
    all_model_results.append(metrics)



🔥 Fine-tuning Base Model: mBERT


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

  ...adapter_model.safetensors: 100%|##########| 3.56MB / 3.56MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

✅ Pushed to HF: https://huggingface.co/shariul-islam/proposed-nourl-distil-mbert



📊 Generating Evaluation Report for Distil-mBERT


              precision    recall  f1-score   support

      normal       0.97      0.94      0.96       498
       promo       0.94      0.89      0.92       342
       smish       0.92      0.97      0.94       561

    accuracy                           0.94      1401
   macro avg       0.94      0.93      0.94      1401
weighted avg       0.94      0.94      0.94      1401


✅ Per-source classification reports saved.
✅ Summary report saved to: ./BERT-Based_Proposed_Version_without_normalization/Distil-mBERT_source_summary_report.csv

✅ Distil-mBERT → Accuracy: 0.9408, F1: 0.9407
✅ All reports saved in ./BERT-Based_Proposed_Version_without_normalization


In [ ]:
import shutil
from google.colab import files
import os

# List of folders to zip and download
folders_to_download = [
    ReportFolderName,
    #"stacking_ensemble_reports"
]

for folder in folders_to_download:
    if os.path.exists(folder):
        zip_filename = f"{folder}.zip"
        # Create zip archive
        shutil.make_archive(folder, 'zip', folder)
        # Download zip
        files.download(zip_filename)
        print(f"✅ Download started for '{zip_filename}'")
    else:
        print(f"⚠️ Folder '{folder}' not found")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started for 'BERT-Based_Proposed_Version_without_normalization.zip'
